# Hungary Demographic & Economic Insights

**Elemzési kérdés:** Hogyan változik Magyarország demográfiai szerkezete, és milyen párhuzamok figyelhetők meg a munkaképes korú népesség, a foglalkoztatás és a gazdasági teljesítmény alakulásában?

**Cél:** KSH-adatok letöltése, tisztítása, összekapcsolása, KPI-k számítása és ábrázolása – saját Python modulokkal.

**Adatforrások**
| Adatkör | Forrás | Mutató |
|---|---|---|
| Demográfia | KSH API (data.ksh.hu) – *HVD - Népesség* | január 1-jei népesség korcsoportonként, fő |
| Foglalkoztatás | KSH STADAT 20.1.1.3 (CSV) | foglalkoztatási ráta, 15–64 évesek, % |
| GDP | KSH API (data.ksh.hu) – *NAMAIN_T01GDP_A* | reál GDP, láncolt volumen (2021-es ref. év), millió Ft |

**Módszertani figyelmeztetés:** a mintaadatok (`data/sample_data/`) **hivatalos KSH-adatok kivonatai** (letöltve: 2026-11-15), nem szintetikus adatok. A népesség év eleji (jan. 1.) állapot, a foglalkoztatás éves átlag – a két mutató időpontja nem azonos. Az együttmozgás nem bizonyít ok-okozati kapcsolatot.

## 2. Környezet és importok

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Projektgyökér megkeresése: felfelé lépkedünk, amíg meg nem találjuk a config/settings.py-t.
# Így mindegy, hogy a kernel a projektgyökérből vagy a notebooks/ mappából indul.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "config" / "settings.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Nem található a projektgyökér (config/settings.py).")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import settings
from src import ksh_client, data_loader, data_cleaning, data_transform, indicators, visualization

pd.set_option("display.width", 140)
print("Projektgyökér:", settings.PROJECT_ROOT)

## 3. Adatgyűjtés

**Kapcsoló:** `USE_SAMPLE_DATA = True` esetén a helyi mintaadatokkal fut a notebook (internet nélkül is).
`False` esetén a következő cella letölti a nyers adatokat a KSH-ból a `data/raw/` mappába.

Dokumentált KSH API-végpontok (https://data.ksh.hu/dokumentacio):
1. `GET /datasets.json` – adatkészletek listája
2. `GET /datasets/{id}/metadata.rdf` – metaadat, benne az állományok `downloadURL`-jei
3. a `downloadURL` – maga az adat (CSV vagy SDMX XML)

In [ ]:
USE_SAMPLE_DATA = True

In [ ]:
if not USE_SAMPLE_DATA:
    try:
        # 1) Adatkészletek listája
        datasets = ksh_client.get_datasets()
        ksh_client.save_json(datasets, settings.RAW_DATASETS_FILE)
        for d in datasets:
            print(d["id"], "|", d["titles"]["hu"])

        # 2) A népesség-adatkészlet állományai
        for dist_id, url in ksh_client.get_distribution_urls(settings.POPULATION_DATASET_ID).items():
            print(dist_id, url)

        # 3) Letöltés a data/raw/ mappába (nyers formában, változtatás nélkül)
        ksh_client.fetch_dataset(settings.POPULATION_DATASET_ID, settings.POPULATION_DISTRIBUTION_ID,
                                 settings.RAW_POPULATION_FILE)
        ksh_client.fetch_dataset(settings.GDP_DATASET_ID, settings.GDP_DISTRIBUTION_ID,
                                 settings.RAW_GDP_FILE)
        ksh_client.download_file(settings.EMPLOYMENT_STADAT_URL, settings.RAW_EMPLOYMENT_FILE)
        print("Letöltés kész.")
    except Exception as error:
        print(f"Letöltési hiba: {error}")
        print("Állítsd USE_SAMPLE_DATA = True értékre, és futtasd újra a notebookot a mintaadatokkal.")
        raise
else:
    print("Mintaadat-mód: nincs letöltés. A források leírása: data/raw/source_metadata.json")

## 4. Adatok betöltése

In [ ]:
if USE_SAMPLE_DATA:
    population_raw = data_loader.load_csv(settings.SAMPLE_POPULATION_FILE)
    employment_raw = data_loader.load_csv(settings.SAMPLE_EMPLOYMENT_FILE)
    gdp_raw = data_loader.load_csv(settings.SAMPLE_GDP_FILE)
else:
    population_raw = data_loader.load_ksh_population(settings.RAW_POPULATION_FILE)
    employment_raw = data_loader.load_ksh_employment_stadat(settings.RAW_EMPLOYMENT_FILE)
    gdp_raw = data_loader.load_ksh_gdp_sdmx(settings.RAW_GDP_FILE)
    gdp_raw = gdp_raw[gdp_raw["year"] >= settings.START_YEAR]

metadata = data_loader.load_json(settings.SOURCE_METADATA_FILE)

for name, df in {"population": population_raw, "employment": employment_raw, "gdp": gdp_raw}.items():
    print(f"\n===== {name} =====")
    display(df.head())
    df.info()
    display(df.describe())

## 5. Adatminőség ellenőrzése

In [ ]:
for name, df in {"population": population_raw, "employment": employment_raw, "gdp": gdp_raw}.items():
    data_cleaning.report_quality(df, name)
    print("Adattípusok:", df.dtypes.astype(str).to_dict())
    print("Mértékegység:", metadata[name]["unit"], "|", metadata[name]["reference_period"])
    print()

## 6. Tisztítás és transzformáció

A tisztítás a `src/data_cleaning.py`, az összekapcsolás a `src/data_transform.py` függvényeivel történik.
**Join:** left join a népességi táblára – minden demográfiai év megmarad, a hiányzó gazdasági adat NaN.
A népességi időszakon kívüli gazdasági évek (pl. 2000–2012 GDP) nem kerülnek az elemzési táblába – ezt a cella kiírja.

In [ ]:
population = data_cleaning.clean_population_data(population_raw)
employment = data_cleaning.clean_economic_data(employment_raw)
gdp = data_cleaning.clean_economic_data(gdp_raw)

merged = data_transform.merge_datasets(population, employment, gdp)
merged.head()

## 7. KPI-k számítása

In [ ]:
result = indicators.calculate_indicators(merged)
kpi_cols = ["year", "working_age_ratio", "elderly_ratio", "ageing_index", "employment_rate",
            "population_15_64_yoy_pct", "gdp_yoy_pct", "gdp_per_capita"]
display(result[kpi_cols].head().round(2))
display(result[kpi_cols].tail().round(2))

## 8. Eredmények exportálása

In [ ]:
settings.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
result.to_csv(settings.PROCESSED_FILE, index=False, encoding="utf-8")
print("Mentve:", settings.PROCESSED_FILE)

## 9. Vizualizáció

Az ábrák az `outputs/charts/` mappába is mentődnek.

In [ ]:
visualization.plot_population_by_age(result, settings.CHART_DIR);

In [ ]:
visualization.plot_ageing_index(result, settings.CHART_DIR);

A foglalkoztatási ráta (%) és a reál GDP (Ft) eltérő mértékegységű, ezért két külön panelen szerepel.
A közös trend **nem bizonyít ok-okozati összefüggést** (pl. 2020: COVID-sokk, mindkettőre hatott).

In [ ]:
visualization.plot_economy_and_employment(result, settings.CHART_DIR);

## 10. Következtetések

Válaszold meg a kérdéseket **a fenti táblák és ábrák alapján** (a válaszokat te írod be):

1. Hogyan változott a 15–64 éves népesség aránya? *(tipp: `working_age_ratio` első és utolsó éve)*
2. Hogyan változott az öregedési index? Átlépte-e a 100-at, és mikor?
3. Mely időszakban volt a legnagyobb változás? *(tipp: `population_15_64_yoy_pct`, `gdp_yoy_pct`)*
4. Mit mutat a foglalkoztatás és a GDP idősora? Hol tér el a kettő?
5. Milyen korlátai vannak az elemzésnek? Gondolj például:
   - a rövid népességi idősorra (2013-tól),
   - az év eleji népesség és az éves átlagos foglalkoztatás eltérő időpontjára,
   - a GDP/fő közelítő számítására (jan. 1-jei népesség a nevező),
   - az előzetes (2024-es) GDP-adatra,
   - korreláció ≠ ok-okozat.

*Válaszok:*

1. …